In [12]:
import torch
torch.manual_seed(0)
dtype = torch.float64  # use double for closer agreement
device = 'cpu'         # CPU for determinism

n_steps = 100
dim = 128
in_dim = 1024

scaling_coef = 0.1

A_init = torch.randn(dim, dim, dtype=dtype, device=device)* scaling_coef
B_init = torch.randn(dim, in_dim, dtype=dtype, device=device)* scaling_coef
C_init = torch.randn(1, dim, dtype=dtype, device=device)* scaling_coef

A = torch.nn.Parameter(A_init, requires_grad=True)
B = torch.nn.Parameter(B_init, requires_grad=True)
C = torch.nn.Parameter(C_init, requires_grad=True)


xs = [torch.randn(in_dim, dtype=dtype, device=device) for _ in range(n_steps)]

# recurrent unroll
def run_recurrent(A,B,C,xs):
    h = torch.zeros(A.size(0), dtype=dtype, device=device)
    for x in xs:
        h = A @ h + B @ x
    y = C @ h
    return y.squeeze()

# convolutional (direct formula)
def run_conv(A, B, C, xs):
    n = len(xs)
    acc = torch.zeros(A.size(0), dtype=A.dtype)

    Apow = torch.eye(A.size(0), dtype=A.dtype)

    # build A^k forward
    powers = [Apow]
    for _ in range(1, n):
        Apow = A @ Apow
        powers.append(Apow)

    # now apply correct alignment
    for i in range(n):
        acc = acc + powers[n-1-i] @ (B @ xs[i])

    y = C @ acc
    return y.squeeze()

y_rec = run_recurrent(A,B,C,xs)
y_conv = run_conv(A,B,C,xs)
loss_rec = (y_rec**2).sum()
loss_conv = (y_conv**2).sum()

loss_rec.backward()
grads_rec = (A.grad.clone(), B.grad.clone(), C.grad.clone())

# clear grads and recompute for conv
A.grad.zero_(); B.grad.zero_(); C.grad.zero_()
loss_conv.backward()
grads_conv = (A.grad.clone(), B.grad.clone(), C.grad.clone())

print('max diff A grad:', (grads_rec[0]-grads_conv[0]).abs().max().item())
print('max diff B grad:', (grads_rec[1]-grads_conv[1]).abs().max().item())
print('max diff C grad:', (grads_rec[2]-grads_conv[2]).abs().max().item())

max diff A grad: 0.010498046875
max diff B grad: 0.000263214111328125
max diff C grad: 0.003570556640625
